# Benchmarks — Throughput & Scaling

This notebook measures:
- Period-finding latency across several `(a, N)` pairs.
- Sampling throughput for Periodic/Product/MPS states.
- (Optional) GPU speedups if CuPy is available.

> Numbers are for **relative** comparison on your machine.

In [ ]:
import time, statistics as stats
from collections import defaultdict
from quantum_hybrid_system import QuantumClassicalHybrid, PeriodicState, ProductState, MatrixProductState

hybrid_cpu = QuantumClassicalHybrid(verbose=False, use_gpu=False)
hybrid_gpu = QuantumClassicalHybrid(verbose=False, use_gpu=True)

# ---- Period finding latency ----
pairs = [(5, 437), (7, 899), (11, 1763), (13, 2467)]
runs = 5
results = []
for a, N in pairs:
    latencies = []
    for _ in range(runs):
        t0 = time.perf_counter()
        res = hybrid_cpu.find_period(a, N, method="auto")
        latencies.append(time.perf_counter() - t0)
    results.append((a, N, res.period, stats.mean(latencies), stats.stdev(latencies) if len(latencies)>1 else 0.0))

print("Period finding (CPU):")
for a, N, r, mu, sd in results:
    print(f"a={a:<2} N={N:<5} r={r:<4} mean={mu*1e3:7.2f} ms  sd={sd*1e3:6.2f} ms")

# ---- Sampling throughput ----
def sample_rate(state, shots=20000):
    t0 = time.perf_counter()
    state.measure(num_shots=shots)
    dt = time.perf_counter() - t0
    return shots / dt

samplers = [
    ("Periodic n=20,r=8", PeriodicState(20, period=8)),
    ("Product  n=24", ProductState(24)),
    ("MPS      n=24,χ=8", MatrixProductState(24, bond_dim=8)),
]

print("\nSampling throughput (samples/sec):")
for name, st in samplers:
    try:
        rate = sample_rate(st, shots=10000)
        print(f"{name:<18} {rate:,.0f}")
    except Exception as e:
        print(f"{name:<18} ERROR: {e}")

# ---- Optional GPU check ----
print("\nGPU check on a single case (falls back if unavailable):")
a, N = 7, 899
cpu = hybrid_cpu.find_period(a, N)
gpu = hybrid_gpu.find_period_gpu(a, N)
print(f"CPU: r={cpu.period} method={cpu.method} time={cpu.time_seconds*1e3:.2f} ms")
print(f"GPU: r={gpu.period} method={gpu.method} time={gpu.time_seconds*1e3:.2f} ms")